# D143 - Transactions and Locks in MySQL

A **transaction** is a group of database operations treated as one logical unit: either all required changes succeed, or none remain. This notebook uses a small account-transfer system so every idea can be run and observed directly.

You will demonstrate COMMIT, ROLLBACK, SAVEPOINT, ACID properties, READ COMMITTED, REPEATABLE READ, row locks, lock waits, and a real deadlock.

## Setup

The local credentials follow D140 and D141: MySQL user root and password root. The next cells recreate transactiondb, so any existing database with that name is removed.

In [ ]:
import threading
import mysql.connector
from mysql.connector import Error

MYSQL_HOSTNAME = "localhost"
MYSQL_PORT = 3306
MYSQL_USERNAME = "root"
MYSQL_PASSWORD = "root"
MYSQL_DATABASE = "transactiondb"

server = mysql.connector.connect(
    host=MYSQL_HOSTNAME, port=MYSQL_PORT,
    user=MYSQL_USERNAME, password=MYSQL_PASSWORD,
)
cursor = server.cursor()
cursor.execute(f"DROP DATABASE IF EXISTS {MYSQL_DATABASE}")
cursor.execute(
    f"CREATE DATABASE {MYSQL_DATABASE} "
    "CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci"
)
cursor.close()
server.close()
print(f"Recreated {MYSQL_DATABASE}.")

### Connection helpers

Isolation and locking involve separate database sessions. Standalone setup and display statements use autocommit, while every teaching transaction begins explicitly and ends with COMMIT or ROLLBACK.

In [ ]:
def new_connection():
    return mysql.connector.connect(
        host=MYSQL_HOSTNAME, port=MYSQL_PORT,
        user=MYSQL_USERNAME, password=MYSQL_PASSWORD,
        database=MYSQL_DATABASE, autocommit=True,
    )


def fetch_all(db, sql, params=()):
    cursor = db.cursor()
    try:
        cursor.execute(sql, params)
        return cursor.fetchall()
    finally:
        cursor.close()


def fetch_value(db, sql, params=()):
    return fetch_all(db, sql, params)[0][0]


def show_accounts(db, label):
    print(f"\n{label}")
    print("id | holder | balance")
    rows = fetch_all(
        db, "SELECT account_id, holder_name, balance FROM accounts ORDER BY account_id"
    )
    for row in rows:
        print(" | ".join(str(value) for value in row))


connection = new_connection()
print("Connected:", connection.is_connected())
print("MySQL version:", connection.get_server_info())

## Create the demonstration data

accounts stores current balances, transfer_log records completed transfers, and isolation_probe provides a harmless value for visibility tests. InnoDB supplies transactional changes and row-level locks.

In [ ]:
cursor = connection.cursor()
try:
    cursor.execute("""
        CREATE TABLE accounts (
            account_id INT UNSIGNED AUTO_INCREMENT PRIMARY KEY,
            holder_name VARCHAR(100) NOT NULL,
            balance DECIMAL(12, 2) NOT NULL,
            CONSTRAINT chk_balance CHECK (balance >= 0)
        ) ENGINE=InnoDB
    """)
    cursor.execute("""
        CREATE TABLE transfer_log (
            transfer_id BIGINT UNSIGNED AUTO_INCREMENT PRIMARY KEY,
            from_account_id INT UNSIGNED NOT NULL,
            to_account_id INT UNSIGNED NOT NULL,
            amount DECIMAL(12, 2) NOT NULL,
            transferred_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
            CONSTRAINT chk_different_accounts CHECK (from_account_id <> to_account_id),
            CONSTRAINT chk_positive_amount CHECK (amount > 0),
            CONSTRAINT fk_transfer_from FOREIGN KEY (from_account_id)
                REFERENCES accounts (account_id),
            CONSTRAINT fk_transfer_to FOREIGN KEY (to_account_id)
                REFERENCES accounts (account_id)
        ) ENGINE=InnoDB
    """)
    cursor.execute("""
        CREATE TABLE isolation_probe (
            probe_id TINYINT UNSIGNED PRIMARY KEY,
            observed_value INT NOT NULL
        ) ENGINE=InnoDB
    """)
    cursor.executemany(
        "INSERT INTO accounts (holder_name, balance) VALUES (%s, %s)",
        [("Asha", 1000.00), ("Bala", 500.00), ("Chen", 250.00)],
    )
    cursor.execute(
        "INSERT INTO isolation_probe (probe_id, observed_value) VALUES (1, 100)"
    )
    connection.commit()
finally:
    cursor.close()
show_accounts(connection, "Initial committed balances")

## Transaction boundaries and ACID

A transaction normally follows: START TRANSACTION, read and lock, validate, change, then COMMIT. An error path uses ROLLBACK.

- **Atomicity:** debit, credit, and log entry succeed or fail together.
- **Consistency:** constraints and validation preserve valid states.
- **Isolation:** concurrent sessions do not freely observe or overwrite unfinished work.
- **Durability:** committed changes survive ordinary failures.

COMMIT makes changes permanent and releases locks. ROLLBACK removes uncommitted changes and releases locks. MySQL DDL generally causes an implicit commit, so keep schema changes outside normal data transactions.

## Demonstration 1 - Commit an atomic transfer

Both accounts are locked in ascending identifier order. Consistent lock ordering reduces deadlock risk. Validation, debit, credit, and logging all occur before one commit.

In [ ]:
def transfer_funds(db, source_id, destination_id, amount):
    if source_id == destination_id or amount <= 0:
        raise ValueError("Use different accounts and a positive amount.")
    cursor = db.cursor()
    try:
        db.start_transaction()
        ordered_ids = sorted((source_id, destination_id))
        cursor.execute("""
            SELECT account_id, balance FROM accounts
            WHERE account_id IN (%s, %s)
            ORDER BY account_id FOR UPDATE
        """, tuple(ordered_ids))
        balances = dict(cursor.fetchall())
        if len(balances) != 2:
            raise ValueError("Both accounts must exist.")
        if balances[source_id] < amount:
            raise ValueError("Insufficient funds.")
        cursor.execute(
            "UPDATE accounts SET balance = balance - %s WHERE account_id = %s",
            (amount, source_id),
        )
        cursor.execute(
            "UPDATE accounts SET balance = balance + %s WHERE account_id = %s",
            (amount, destination_id),
        )
        cursor.execute("""
            INSERT INTO transfer_log (from_account_id, to_account_id, amount)
            VALUES (%s, %s, %s)
        """, (source_id, destination_id, amount))
        db.commit()
        print(f"Committed transfer of {amount:.2f}.")
    except Exception:
        db.rollback()
        print("Transaction rolled back.")
        raise
    finally:
        cursor.close()


show_accounts(connection, "Before transfer")
transfer_funds(connection, 1, 2, 125.00)
show_accounts(connection, "After committed transfer")

## Demonstration 2 - Roll back after failure

The example raises an error after the debit. The same session sees its own uncommitted change, but ROLLBACK restores the committed balance.

In [ ]:
show_accounts(connection, "Before deliberate failure")
cursor = connection.cursor()
try:
    connection.start_transaction()
    cursor.execute("UPDATE accounts SET balance = balance - 50 WHERE account_id = 1")
    cursor.execute("SELECT balance FROM accounts WHERE account_id = 1")
    print("Uncommitted balance in this session:", cursor.fetchone()[0])
    raise RuntimeError("Simulated failure before crediting the destination")
except RuntimeError as error:
    print(error)
    connection.rollback()
    print("Rolled back the whole transaction.")
finally:
    cursor.close()
show_accounts(connection, "After rollback")

## Demonstration 3 - Savepoint and partial rollback

A savepoint marks a position inside a transaction. The transfer remains, while an optional fee added after the savepoint is undone. ROLLBACK TO SAVEPOINT does not finish the transaction; COMMIT is still required.

In [ ]:
cursor = connection.cursor()
try:
    connection.start_transaction()
    cursor.execute(
        "SELECT account_id FROM accounts WHERE account_id IN (2, 3) "
        "ORDER BY account_id FOR UPDATE"
    )
    cursor.fetchall()
    cursor.execute("UPDATE accounts SET balance = balance - 25 WHERE account_id = 2")
    cursor.execute("UPDATE accounts SET balance = balance + 25 WHERE account_id = 3")
    cursor.execute("SAVEPOINT before_optional_fee")
    cursor.execute("UPDATE accounts SET balance = balance - 5 WHERE account_id = 2")
    cursor.execute("ROLLBACK TO SAVEPOINT before_optional_fee")
    cursor.execute("""
        INSERT INTO transfer_log (from_account_id, to_account_id, amount)
        VALUES (2, 3, 25)
    """)
    connection.commit()
    print("Core transfer committed; optional fee was rolled back.")
except Exception:
    connection.rollback()
    raise
finally:
    cursor.close()
show_accounts(connection, "After savepoint demonstration")

## Demonstration 4 - Isolation levels

READ COMMITTED can return a newly committed value on the second read. Under MySQL's default REPEATABLE READ, ordinary reads in one transaction use the same snapshot.

In [ ]:
def demonstrate_isolation(level):
    reader, writer = new_connection(), new_connection()
    try:
        cursor = reader.cursor()
        cursor.execute(f"SET TRANSACTION ISOLATION LEVEL {level}")
        cursor.close()
        reader.start_transaction()
        first = fetch_value(
            reader, "SELECT observed_value FROM isolation_probe WHERE probe_id = 1"
        )
        writer.start_transaction()
        cursor = writer.cursor()
        cursor.execute(
            "UPDATE isolation_probe SET observed_value = observed_value + 1 WHERE probe_id = 1"
        )
        cursor.close()
        writer.commit()
        second = fetch_value(
            reader, "SELECT observed_value FROM isolation_probe WHERE probe_id = 1"
        )
        reader.commit()
        print(f"{level}: first={first}, second={second}, same={first == second}")
    finally:
        if reader.in_transaction:
            reader.rollback()
        if writer.in_transaction:
            writer.rollback()
        reader.close()
        writer.close()


demonstrate_isolation("READ COMMITTED")
demonstrate_isolation("REPEATABLE READ")

## Locks and a visible lock wait

SELECT ... FOR UPDATE takes exclusive row locks for a later change. UPDATE and DELETE lock automatically. Locks normally remain until COMMIT or ROLLBACK. Plain SELECT usually reads a snapshot without locking.

Session A locks one account. Session B attempts a conflicting update with a two-second timeout, then retries after Session A releases the lock.

In [ ]:
session_a, session_b = new_connection(), new_connection()
cursor_a, cursor_b = session_a.cursor(), session_b.cursor()
try:
    session_a.start_transaction()
    cursor_a.execute("SELECT balance FROM accounts WHERE account_id = 1 FOR UPDATE")
    print("Session A locked account 1; balance:", cursor_a.fetchone()[0])
    cursor_b.execute("SET SESSION innodb_lock_wait_timeout = 2")
    session_b.start_transaction()
    try:
        print("Session B attempts a conflicting update...")
        cursor_b.execute("UPDATE accounts SET balance = balance WHERE account_id = 1")
    except Error as error:
        print(f"Session B error {error.errno}: {error.msg}")
        session_b.rollback()
    session_a.commit()
    print("Session A committed and released its lock.")
    session_b.start_transaction()
    cursor_b.execute("UPDATE accounts SET balance = balance WHERE account_id = 1")
    session_b.commit()
    print("Session B retry succeeded.")
finally:
    if session_a.in_transaction: session_a.rollback()
    if session_b.in_transaction: session_b.rollback()
    cursor_a.close(); cursor_b.close()
    session_a.close(); session_b.close()

## Demonstration 6 - Create and handle a deadlock

Worker A locks row 1 and requests row 2. Worker B locks row 2 and requests row 1. A barrier creates the cycle reliably. InnoDB should commit one transaction and cancel the other with error 1213. Applications should retry the complete cancelled unit of work when it is safe.

In [ ]:
barrier = threading.Barrier(2)
deadlock_results = []
results_guard = threading.Lock()


def deadlock_worker(name, first_id, second_id):
    db = new_connection()
    cursor = db.cursor()
    try:
        cursor.execute("SET SESSION innodb_lock_wait_timeout = 5")
        db.start_transaction()
        cursor.execute(
            "SELECT account_id FROM accounts WHERE account_id = %s FOR UPDATE",
            (first_id,),
        )
        cursor.fetchone()
        barrier.wait(timeout=5)
        cursor.execute(
            "SELECT account_id FROM accounts WHERE account_id = %s FOR UPDATE",
            (second_id,),
        )
        cursor.fetchone()
        db.commit()
        result = f"{name}: committed"
    except Error as error:
        db.rollback()
        result = f"{name}: rolled back with MySQL error {error.errno} - {error.msg}"
    except threading.BrokenBarrierError:
        db.rollback()
        result = f"{name}: barrier failed; rolled back"
    finally:
        cursor.close(); db.close()
        with results_guard:
            deadlock_results.append(result)


worker_a = threading.Thread(target=deadlock_worker, args=("Worker A", 1, 2))
worker_b = threading.Thread(target=deadlock_worker, args=("Worker B", 2, 1))
worker_a.start(); worker_b.start()
worker_a.join(timeout=10); worker_b.join(timeout=10)
for result in sorted(deadlock_results):
    print(result)
print("Expected: one commit and one MySQL error 1213.")

## Practical checklist

- Keep the unit of work and lock duration as small as correctness permits.
- Validate while holding the locks that protect the decision.
- Use indexed predicates and lock resources in a consistent order.
- Commit once after all required operations succeed; roll back every failure path.
- Keep user input, network calls, and slow computation outside transactions.
- Treat lock timeouts and deadlocks as normal possible outcomes.
- Retry only safe, repeatable operations with bounded attempts and backoff.
- Keep DDL that causes implicit commits separate from transactional DML.

In [ ]:
show_accounts(connection, "Final account balances")
print("Transfer log:")
for row in fetch_all(connection, """
    SELECT transfer_id, from_account_id, to_account_id, amount, transferred_at
    FROM transfer_log ORDER BY transfer_id
"""):
    print(row)
if connection.in_transaction:
    connection.rollback()
connection.close()
print("Connection closed.")